In [ ]:
import json
import re
import unicodedata
import hashlib
from collections import defaultdict, Counter
from pathlib import Path

import pandas as pd


def norm(value):
    value = unicodedata.normalize('NFKD', str(value or '')).casefold()
    value = ''.join(c for c in value if not unicodedata.combining(c))
    return ' '.join(re.sub(r'[^\w]+', ' ', value).split())


def name_key(name, brand):
    name, brand = norm(name), norm(brand)
    return name[len(brand) + 1:] if brand and name.startswith(brand + ' ') else name


def tokens(values):
    return list(dict.fromkeys(norm(v).replace(' ', '_') for v in values
                              if norm(v) not in {'', 'unknown', 'nan', 'none', 'null', 'n a'}))


def write_csv(frame, path):
    frame.to_csv(path, index=False, encoding='utf-8-sig', lineterminator='\n')


def preprocess(v2_jsonl, existing_dir, output_dir):
    v2_jsonl, existing_dir, output_dir = map(Path, (v2_jsonl, existing_dir, output_dir))
    output_dir.mkdir(parents=True, exist_ok=True)
    audit = output_dir / 'audit'
    audit.mkdir(exist_ok=True)
    old = pd.read_csv(existing_dir / 'perfumes_content.csv', keep_default_na=False)
    mapping_old = pd.read_csv(existing_dir / 'perfume_mapping.csv', keep_default_na=False)
    atr = pd.read_csv(existing_dir / 'atrafshan_perfumes.csv', keep_default_na=False)
    ratings = pd.read_csv(existing_dir / 'user_ratings.csv', keep_default_na=False)
    conflicts = pd.read_csv(existing_dir / 'audit/atrafshan_metadata_conflicts.csv', keep_default_na=False)
    bad_ids = set(conflicts.perfume_id)
    old_by_id = old.set_index('perfume_id').to_dict('index')
    items, edges, seen = [], [], set()
    source_hash = hashlib.sha256()
    with v2_jsonl.open('rb') as source:
        for source_row, line in enumerate(source, 1):
            source_hash.update(line)
            r = json.loads(line)
            pid = 'fr:' + str(r['id'])
            if pid in seen:
                raise ValueError('Duplicate Fragrantica ID: ' + pid)
            seen.add(pid)
            previous = old_by_id.get(pid, {})
            notes = r.get('notes') or {}
            tiers = notes.get('tiered') or {}
            top, middle, base = [tokens(x.get('name', '') for x in tiers.get(t, []))
                                 for t in ('top', 'middle', 'base')]
            flat = tokens(x.get('name', '') for x in notes.get('flat', []))
            accords = tokens(x.get('name', '') for x in r.get('accords', []))
            all_notes = list(dict.fromkeys(top + middle + base + flat))
            features = ['note_' + x for x in all_notes] + ['accord_' + x for x in accords]
            content_source = 'v2'
            # Keep usable legacy features when V2 has none.
            if not features and previous.get('content_text', ''):
                features = str(previous['content_text']).split()
                top, middle, base, accords = [json.loads(previous[c]) for c in
                    ('top_notes', 'middle_notes', 'base_notes', 'main_accords')]
                content_source = 'legacy_fallback'
            rating = r.get('rating') or {}
            histogram = rating.get('histogram') or []
            count = sum(x.get('count', 0) for x in histogram) if histogram else None
            items.append(dict(
                perfume_id=pid, perfume_name=r['name'], brand=r['brand'],
                name_normalized=norm(r['name']), brand_normalized=norm(r['brand']),
                gender={'male': 'men', 'female': 'women'}.get(r.get('gender'), r.get('gender') or ''),
                release_year=r.get('year'), country=previous.get('country', ''),
                top_notes=json.dumps(top), middle_notes=json.dumps(middle),
                base_notes=json.dumps(base), main_accords=json.dumps(accords),
                content_text=' '.join(features), content_eligible=bool(features),
                avg_rating=rating.get('average'), rating_count=count, url=r.get('url', ''),
                raw_display_name=r['name'], description=r.get('description') or '',
                source_dataset=v2_jsonl.name, source_row=source_row,
                supplement_source_row=previous.get('source_row', ''),
                flat_notes=json.dumps(flat), content_source=content_source))
            # Evaluation labels stay outside content features.
            for e in (r.get('similar') or {}).get('reminds_me_of', []):
                if e.get('id') is None:
                    continue
                edges.append(dict(query_perfume_id=pid, target_perfume_id='fr:' + str(e['id']),
                                  up_votes=e.get('up_votes'), down_votes=e.get('down_votes'),
                                  source_row=source_row))
    v2_ids = set(seen)
    # Retain the four legacy-only items, using the same stable IDs.
    for r in old.to_dict('records'):
        if r['perfume_id'] not in seen:
            r.update(flat_notes='[]', content_source='legacy_only')
            items.append(r)
    content = pd.DataFrame(items)
    exact = defaultdict(list)
    for r in items:
        exact[(norm(r['brand']), name_key(r['perfume_name'], r['brand']))].append(r)
    previous_map = dict(zip(mapping_old.loc[mapping_old.match_status.eq('matched_exact'), 'atrafshan_id'],
                            mapping_old.loc[mapping_old.match_status.eq('matched_exact'), 'fragrantica_id']))
    candidates, decisions = {}, {}
    for r in atr.to_dict('records'):
        aid = r['perfume_id']
        matches = exact[(norm(r['brand']), name_key(r['perfume_name'], r['brand']))]
        if aid in previous_map:
            candidates[aid] = previous_map[aid]
            decisions[aid] = ('matched_exact', 'retained_verified_site_id')
        elif aid in bad_ids:
            decisions[aid] = ('review_required', 'inconsistent_source_metadata')
        elif len(matches) != 1:
            decisions[aid] = ('review_required', 'ambiguous_exact_name') if matches else ('unmatched', 'no_exact_brand_and_name')
        else:
            f = matches[0]
            ay, fy = pd.to_numeric(r['release_year'], errors='coerce'), pd.to_numeric(f['release_year'], errors='coerce')
            if pd.notna(ay) and pd.notna(fy) and ay != fy:
                decisions[aid] = ('review_required', 'release_year_conflict')
            else:
                candidates[aid] = f['perfume_id']
                decisions[aid] = ('matched_exact', 'unique_normalized_brand_and_name')
    target_counts = Counter(candidates.values())
    for aid, fid in list(candidates.items()):
        if target_counts[fid] > 1 and aid not in previous_map:
            candidates.pop(aid)
            decisions[aid] = ('review_required', 'multiple_atrafshan_ids_for_one_fragrantica_id')
    mapping = pd.DataFrame([dict(atrafshan_id=r['perfume_id'], atrafshan_name=r['perfume_name'],
        atrafshan_brand=r['brand'], fragrantica_id=candidates.get(r['perfume_id'], ''),
        match_status=decisions[r['perfume_id']][0], reason=decisions[r['perfume_id']][1])
        for r in atr.to_dict('records')])
    matched = mapping[mapping.match_status.eq('matched_exact')]
    linked = ratings.merge(matched[['atrafshan_id', 'fragrantica_id']], left_on='perfume_id',
                           right_on='atrafshan_id', validate='many_to_one').drop(columns='atrafshan_id')
    eligible_ids = set(content.loc[content.content_eligible, 'perfume_id'])
    edge_df = pd.DataFrame(edges)
    valid = (edge_df.query_perfume_id.isin(eligible_ids) & edge_df.target_perfume_id.isin(eligible_ids)
             & edge_df.query_perfume_id.ne(edge_df.target_perfume_id))
    for c in ('up_votes', 'down_votes'):
        edge_df[c] = pd.to_numeric(edge_df[c], errors='coerce')
    edge_df['net_votes'] = edge_df.up_votes - edge_df.down_votes
    edge_df['total_votes'] = edge_df.up_votes + edge_df.down_votes
    edge_df['eligible_positive'] = valid & edge_df.up_votes.gt(edge_df.down_votes) & edge_df.total_votes.ge(5)
    edge_df['in_legacy_catalog'] = edge_df.query_perfume_id.isin(old_by_id) & edge_df.target_perfume_id.isin(old_by_id)
    duplicate_edges = edge_df.duplicated(['query_perfume_id', 'target_perfume_id'], keep=False)
    write_csv(edge_df[~valid | duplicate_edges], audit / 'similarity_edges_review.csv')
    # Strongest observation wins for repeated directed edges; reverse edges remain distinct.
    edge_df = edge_df[valid].sort_values('total_votes', ascending=False).drop_duplicates(['query_perfume_id', 'target_perfume_id'])
    for frame, filename in [(content, 'perfumes_content.csv'), (ratings, 'user_ratings.csv'),
                            (atr, 'atrafshan_perfumes.csv'), (mapping, 'perfume_mapping.csv'),
                            (linked, 'user_ratings_linked.csv'), (edge_df, 'similarity_evaluation.csv')]:
        write_csv(frame, output_dir / filename)
    assert content.perfume_id.is_unique
    assert matched.fragrantica_id.is_unique
    assert set(linked.fragrantica_id) <= set(content.perfume_id)
    assert not ratings.duplicated(['user_id', 'perfume_id']).any()
    assert ratings.scent_rating.between(1, 10).all()
    assert all(candidates[k] == v for k, v in previous_map.items())
    assert len(content) == len(v2_ids | set(old_by_id))
    # Check the serialized files as well.
    exported = pd.read_csv(output_dir / 'perfumes_content.csv', keep_default_na=False)
    assert len(exported) == len(content) and exported.perfume_id.is_unique
    for c in ('top_notes', 'middle_notes', 'base_notes', 'flat_notes', 'main_accords'):
        assert exported[c].map(lambda x: isinstance(json.loads(x), list)).all()
    user_counts = linked.groupby('user_id').size()
    summary = dict(v2_source_sha256=source_hash.hexdigest(), v2_items=len(v2_ids),
        legacy_items=len(old), overlap_items=len(v2_ids & set(old_by_id)),
        union_items=len(content), content_eligible_items=int(content.content_eligible.sum()),
        content_sources=content.content_source.value_counts().to_dict(),
        flat_note_items=int(content.flat_notes.ne('[]').sum()),
        mapping_counts=mapping.match_status.value_counts().to_dict(),
        old_verified_mappings_retained=len(previous_map), linked_ratings=len(linked),
        linked_users=len(user_counts), linked_users_ge3=int(user_counts.ge(3).sum()),
        linked_rated_items=int(linked.fragrantica_id.nunique()),
        ratings_unchanged=len(ratings), similarity_raw_edges=len(edges),
        similarity_valid_edges=len(edge_df), similarity_positive_edges=int(edge_df.eligible_positive.sum()),
        similarity_legacy_positive_edges=int((edge_df.eligible_positive & edge_df.in_legacy_catalog).sum()),
        evaluation_default='directed edges, up_votes > down_votes, total_votes >= 5; threshold requires validation',
        trained_or_evaluated=False, validation='passed')
    (output_dir / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(json.dumps(summary, indent=2))
    return summary
